# 🎯 IT ENGINEER ATS SYSTEM — Training Notebook

This notebook trains a model using YOUR rejection labels on YOUR 10 PDFs

## What this does:
✅ Scans data/ folder for your 10 PDFs
✅ Creates CSV template for rejection counts
✅ You fill in how many times each resume was rejected (1, 2, 3, or 4)
✅ Trains Random Forest model
✅ Saves trained model for the Streamlit app

## CELL 1: Install Dependencies

In [ ]:
!pip install pdfplumber spacy scikit-learn numpy pandas matplotlib tqdm joblib openpyxl --quiet
!python -m spacy download en_core_web_sm --quiet
print('✅ All dependencies installed.')

## CELL 2: Imports & Configuration

In [ ]:
import os, re, glob, json, datetime, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pdfplumber
import spacy
import joblib

from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Configuration - ADAPTED TO YOUR STRUCTURE
DATA_FOLDER   = 'data'          # Your data folder with PDFs
OUTPUT_FOLDER = 'ats_output'    # Already exists in your structure
MODEL_DIR     = 'ats_models'    # Already exists in your structure
LABELS_FILE   = 'resume_labels.csv'

os.makedirs(OUTPUT_FOLDER, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

# Load NLP Models
print('Loading NLP models...')
nlp = spacy.load('en_core_web_sm')
sent_model = SentenceTransformer('all-MiniLM-L6-v2')
print('✅ NLP models loaded.')
print(f'✅ Data folder: {os.path.abspath(DATA_FOLDER)}')
print(f'✅ Output folder: {os.path.abspath(OUTPUT_FOLDER)}')
print(f'✅ Model folder: {os.path.abspath(MODEL_DIR)}')

## CELL 3: PDF Parsing Functions

In [ ]:
def extract_pdf_text(pdf_path: str) -> dict:
    """Extract text and metadata from PDF resume."""
    raw_text, page_count, has_tables = '', 0, False
    try:
        with pdfplumber.open(pdf_path) as pdf:
            page_count = len(pdf.pages)
            for page in pdf.pages:
                text = page.extract_text()
                if text:
                    raw_text += text + '\n'
                if page.extract_tables():
                    has_tables = True
    except Exception as e:
        print(f'  ⚠️ Failed to parse {os.path.basename(pdf_path)}: {e}')
    return {
        'file_name': os.path.basename(pdf_path),
        'file_path': pdf_path,
        'raw_text': raw_text,
        'page_count': page_count,
        'has_tables': has_tables,
    }


def clean_text(text: str) -> str:
    """Clean and normalize text."""
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)
    text = re.sub(r'[\x80-\xFF]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text.lower()


def is_it_engineer_resume(text: str) -> bool:
    """Check if resume is for IT/Software Engineer role."""
    it_keywords = [
        r'\b(software engineer|backend|frontend|full.?stack|developer)\b',
        r'\b(java|python|c\+\+|javascript|golang|rust)\b',
        r'\b(systems|infrastructure|devops|cloud)\b',
        r'\b(aws|azure|gcp|kubernetes|docker)\b',
    ]
    matches = sum(1 for kw in it_keywords if re.search(kw, text, re.IGNORECASE))
    return matches >= 2

print('✅ PDF parsing functions ready.')

## CELL 4: Load All Resumes

In [ ]:
def load_all_resumes(data_folder: str) -> pd.DataFrame:
    """Load and parse all IT Engineer PDFs from folder."""
    pdf_paths = glob.glob(os.path.join(data_folder, '*.pdf')) + \
                glob.glob(os.path.join(data_folder, '**', '*.pdf'), recursive=True)
    pdf_paths = list(set(pdf_paths))

    if not pdf_paths:
        raise FileNotFoundError(f'No PDFs found in {data_folder}')

    print(f'Found {len(pdf_paths)} PDFs. Parsing...')
    records = []
    for path in tqdm(pdf_paths):
        rec = extract_pdf_text(path)
        rec['clean_text'] = clean_text(rec['raw_text'])
        rec['word_count'] = len(rec['clean_text'].split())
        rec['is_it_engineer'] = is_it_engineer_resume(rec['clean_text'])
        records.append(rec)

    df = pd.DataFrame(records)
    
    # Filter empty resumes
    df = df[df['word_count'] >= 20].reset_index(drop=True)
    
    # Filter IT Engineers only
    it_count = df['is_it_engineer'].sum()
    print(f'✅ Loaded {len(df)} IT Engineer resumes')
    df = df[df['is_it_engineer']].reset_index(drop=True)
    
    return df

print('\n📁 Loading resumes from your data folder...')
resume_df = load_all_resumes(DATA_FOLDER)
print(f'\n📋 Resume Summary:')
print(f'   Total: {len(resume_df)}')
print(f'   Avg words: {resume_df["word_count"].mean():.0f}')
print(f'\n📄 Resume Files Found:')
for i, fname in enumerate(resume_df['file_name'], 1):
    print(f'   {i}. {fname}')

## CELL 5: Create Labels File Template (YOU FILL THIS IN)

In [ ]:
# Create a template CSV for you to fill in
labels_df = resume_df[['file_name']].copy()
labels_df['rejection_count'] = 0  # Default value (CHANGE THIS)

# Save to CSV
labels_path = os.path.join(OUTPUT_FOLDER, LABELS_FILE)
labels_df.to_csv(labels_path, index=False)

print(f'✅ LABELS FILE CREATED: {labels_path}')
print('\n🔄 INSTRUCTIONS:')
print('1. Open the CSV file with Excel or text editor')
print('2. For each resume, fill in rejection_count (1, 2, 3, or 4)')
print('   - 1 = Rejected once')
print('   - 2 = Rejected twice')
print('   - 3 = Rejected 3 times')
print('   - 4 = Rejected 4+ times')
print('3. Save the file')
print('4. Run the next cell to load your labels')
print('\nTemplate created:')
print(labels_df.to_string(index=False))

## CELL 6: Load Your Rejection Labels

In [ ]:
labels_path = os.path.join(OUTPUT_FOLDER, LABELS_FILE)

if not os.path.exists(labels_path):
    print(f'❌ Labels file not found: {labels_path}')
    print('Please run Cell 5 and fill in the CSV file')
else:
    labels_df = pd.read_csv(labels_path)
    
    # Merge with resume data
    resume_df = resume_df.merge(labels_df, on='file_name', how='left')
    
    print('✅ Labels loaded!')
    print(f'\n📊 Rejection Distribution:')
    print(resume_df['rejection_count'].value_counts().sort_index())
    print(f'\nResumes with labels:')
    print(resume_df[['file_name', 'rejection_count']])

## CELL 7: Define IT Skills

In [ ]:
# Comprehensive IT Engineering skills
IT_SKILLS = {
    'Languages': [
        'python', 'java', 'c++', 'c#', 'javascript', 'go', 'golang', 'rust',
        'ruby', 'php', 'scala', 'kotlin', 'swift', 'typescript'
    ],
    'Databases': [
        'sql', 'postgresql', 'postgres', 'mysql', 'mongodb', 'redis',
        'cassandra', 'elasticsearch', 'dynamodb', 'firestore'
    ],
    'Cloud & DevOps': [
        'aws', 'azure', 'gcp', 'docker', 'kubernetes', 'k8s', 'terraform',
        'ansible', 'jenkins', 'cicd', 'ci/cd', 'github', 'gitlab', 'bitbucket'
    ],
    'Frameworks': [
        'spring', 'django', 'flask', 'fastapi', 'node', 'nodejs',
        'react', 'vue', 'angular', 'express'
    ],
    'Data & ML': [
        'tensorflow', 'keras', 'scikit-learn', 'scikit', 'pandas', 'numpy',
        'apache spark', 'spark', 'hadoop', 'kafka'
    ],
    'Architecture': [
        'microservices', 'api', 'rest', 'restful', 'graphql', 'grpc',
        'distributed systems', 'system design', 'design patterns'
    ],
}

# Flatten for easy lookup
ALL_IT_SKILLS = {skill: category for category, skills in IT_SKILLS.items() for skill in skills}

print(f'✅ Defined {len(ALL_IT_SKILLS)} IT skills across {len(IT_SKILLS)} categories')
for cat, skills in IT_SKILLS.items():
    print(f'   {cat}: {len(skills)} skills')

## CELL 8: Extract Features & Build Training Data

In [ ]:
def extract_it_skills(text: str) -> list:
    """Extract IT skills found in resume."""
    found_skills = []
    for skill in ALL_IT_SKILLS.keys():
        if re.search(r'\b' + skill + r'\b', text, re.IGNORECASE):
            found_skills.append(skill)
    return list(set(found_skills))


def extract_years_experience(text: str) -> float:
    """Extract years of experience."""
    patterns = [
        r'(\d+)\+?\s*years?\s+of\s+experience',
        r'(\d+)\s*years?\s+(?:in|of)\s+',
    ]
    for pattern in patterns:
        matches = re.findall(pattern, text, re.IGNORECASE)
        if matches:
            return float(max(matches))
    return 0.0

print('Extracting features for all IT resumes...')
features_list = []

for idx, row in tqdm(resume_df.iterrows(), total=len(resume_df)):
    text = row['clean_text']
    
    # Extract skills
    found_skills = extract_it_skills(text)
    
    # Count by category
    skill_counts = {cat: 0 for cat in IT_SKILLS.keys()}
    for skill in found_skills:
        cat = ALL_IT_SKILLS[skill]
        skill_counts[cat] += 1
    
    features_list.append({
        'file_name': row['file_name'],
        'total_skills_found': len(found_skills),
        'skills': found_skills,
        'years_exp': extract_years_experience(text),
        'word_count': row['word_count'],
        **skill_counts,
        'rejection_count': row.get('rejection_count', 0),
    })

features_df = pd.DataFrame(features_list)
print(f'\n✅ Features extracted for {len(features_df)} resumes')
print(f'\nFeature Summary:')
print(features_df.describe())

## CELL 9: Train Random Forest Classifier (CORE - MOST IMPORTANT)

In [ ]:
# Prepare training data
feature_cols = [col for col in features_df.columns if col not in ['file_name', 'skills', 'rejection_count']]
X = features_df[feature_cols].fillna(0).values
y = features_df['rejection_count'].values

print(f'Training features: {feature_cols}')
print(f'Target distribution:')
print(pd.Series(y).value_counts().sort_index())

# Train/test split
if len(X) >= 5:
    try:
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y
        )
    except ValueError:
        X_train, X_test, y_train, y_test = X, X, y, y
else:
    X_train, X_test, y_train, y_test = X, X, y, y

print(f'\nTrain size: {len(X_train)}, Test size: {len(X_test)}')

# Train classifier
print('\n🤖 Training Random Forest Classifier...')
rf_classifier = RandomForestClassifier(
    n_estimators=100,
    max_depth=6,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)
rf_classifier.fit(X_train, y_train)

# Evaluate
train_acc = rf_classifier.score(X_train, y_train)
test_acc = rf_classifier.score(X_test, y_test)

print(f'\n📊 MODEL PERFORMANCE:')
print(f'   Train Accuracy: {train_acc:.4f}')
print(f'   Test Accuracy:  {test_acc:.4f}')

if len(X) >= 5:
    y_pred = rf_classifier.predict(X_test)
    print(f'\n   Classification Report:')
    print(classification_report(y_test, y_pred, zero_division=0))

# Feature importance
importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf_classifier.feature_importances_
}).sort_values('importance', ascending=False)

print(f'\n   Top Features (for rejection prediction):')
for idx, row in importance_df.head(10).iterrows():
    print(f'      {row["feature"]}: {row["importance"]:.4f}')

# Save model
model_path = os.path.join(MODEL_DIR, 'rf_it_engineer_classifier.pkl')
joblib.dump({
    'model': rf_classifier,
    'feature_cols': feature_cols,
    'it_skills': IT_SKILLS,
    'all_it_skills': ALL_IT_SKILLS,
}, model_path)
print(f'\n✅ Model saved → {model_path}')

## CELL 10: Analyze Skill Gaps by Rejection Count

In [ ]:
print('📊 SKILL GAP ANALYSIS BY REJECTION COUNT\n')
print('='*70)

for rejection_num in sorted(features_df['rejection_count'].unique()):
    subset = features_df[features_df['rejection_count'] == rejection_num]
    
    print(f'\n🔴 REJECTION COUNT: {rejection_num} ({len(subset)} resumes)')
    print('-'*70)
    
    # Average skills by category
    print('Avg Skills by Category:')
    for cat in IT_SKILLS.keys():
        avg_skills = subset[cat].mean()
        print(f'  {cat:<20}: {avg_skills:.1f} skills/resume')
    
    # Most common missing skills
    all_found = []
    for skills in subset['skills']:
        all_found.extend(skills)
    
    from collections import Counter
    if all_found:
        skill_counts = Counter(all_found)
        print(f'\n  Most common skills in this group:')
        for skill, count in skill_counts.most_common(5):
            print(f'    • {skill}: {count} resumes')

print('\n' + '='*70)

## CELL 11: Save Summary Report

In [ ]:
report_path = os.path.join(OUTPUT_FOLDER, 'training_summary.txt')
with open(report_path, 'w') as f:
    f.write('IT ENGINEER ATS - SUPERVISED LEARNING REPORT\n')
    f.write('='*70 + '\n')
    f.write(f'Training Date: {datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}\n\n')
    
    f.write('TRAINING DATA SUMMARY:\n')
    f.write(f'  Total Resumes: {len(features_df)}\n')
    f.write(f'  Avg Skills Found: {features_df["total_skills_found"].mean():.1f}\n')
    f.write(f'  Avg Years Experience: {features_df["years_exp"].mean():.1f}\n')
    f.write(f'\n  Rejection Distribution:\n')
    for rej_count in sorted(features_df['rejection_count'].unique()):
        count = len(features_df[features_df['rejection_count'] == rej_count])
        pct = 100 * count / len(features_df)
        f.write(f'    {rej_count} rejection(s): {count} resumes ({pct:.1f}%)\n')
    
    f.write(f'\nMODEL PERFORMANCE:\n')
    f.write(f'  Train Accuracy: {train_acc:.4f}\n')
    f.write(f'  Test Accuracy: {test_acc:.4f}\n')
    
    f.write(f'\nTOP PREDICTIVE FEATURES:\n')
    for idx, row in importance_df.head(10).iterrows():
        f.write(f'  {row["feature"]}: {row["importance"]:.4f}\n')

print(f'✅ Report saved → {report_path}')
print(f'\n🎉 TRAINING COMPLETE!')
print(f'\nFiles created:')
print(f'  ✅ {model_path}')
print(f'  ✅ {report_path}')
print(f'\nYour model is ready to use in the Streamlit app!')